In [2]:
from ucimlrepo import fetch_ucirepo 
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
%matplotlib inline
DATA_DIR = Path('../src/shared_util/data')
os.makedirs(DATA_DIR, exist_ok=True)

# fetch dataset 
diabetes_130_us_hospitals_for_years_1999_2008 = fetch_ucirepo(id=296) 
  
# data (as pandas dataframes) 
X: pd.DataFrame = diabetes_130_us_hospitals_for_years_1999_2008.data.features  # type: ignore
y :pd.DataFrame = diabetes_130_us_hospitals_for_years_1999_2008.data.targets.copy()  # type: ignore

y['readmitted'] = y['readmitted'].astype('string')

y['target'] = (y['readmitted'] == '<30').astype(int)


# drop weight and payer_code missing and un informative
X_0 = X.drop(columns=['weight', 'payer_code'])


# fix diag column types
X_0['diag_1'] = X_0['diag_1'].astype('string')
X_0['diag_2'] = X_0['diag_2'].astype('string')
X_0['diag_3'] = X_0['diag_3'].astype('string')

# fix specialty column type
X_0['medical_specialty'] = X_0['medical_specialty'].astype('string')

# fix race
X_0['race'] = X_0['race'].astype('string')

# fix age
X_0['age'] = X_0['age'].astype('string')

X_0['change'] = X_0['change'].astype('string')


/home/mike/Git-projects/Eaton_633_Project/.venv/lib/python3.13/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [3]:
# drop the newborn rows
newborn = (X_0['admission_type_id'] == 4) | (X_0['discharge_disposition_id'] == 10)

_pre = len(X_0)


X_1 = X_0.loc[~newborn].reset_index(drop=True)
y   = y.loc[~newborn].reset_index(drop=True)
print(f'dropped: {_pre - len(X_1)}')

dropped: 16


In [4]:
# Drop rows with invalid gender
valid_gender = (X_1['gender'] == 'Male') | (X_1['gender'] == 'Female')
_pre = len(X_1)
X_2 = X_1.loc[valid_gender].reset_index(drop=True)
y   = y.loc[valid_gender].reset_index(drop=True)
print(f'dropped: {_pre - len(X_2)}')

dropped: 3


In [5]:
# drop rows that show a patient expired

# from mapping info
_c = 'discharge_disposition_id'
exp_mask = (X_2[_c] == 11) | ((X_2[_c] > 18) & (X_2[_c] < 21))

_pre = len(X_2)
X_3 = X_2.loc[~exp_mask].reset_index(drop=True)
y   = y.loc[~exp_mask].reset_index(drop=True)
print(f'dropped: {_pre - len(X_3)}')

dropped: 1652


After examining the numercial features further, the data showed that number_outpatient, number_emergency, and number_inpatient were heavily right skewed with the vast majority of data consisting of 0 of each visit type.

However, number_inpatient has shown to be a dominant factor in model predictions.

After testing combining the patient visit columns, recall dropped slightly in our random forest model without any increase in ROC_AUC or other metric changes so this seems like a poor decision - loss of information with no gain